<div
     style="padding: 20px;
            color: white;
            font-size: 250%;
            text-align: center;
            display: fill;
            border-radius: 5px;
            background-color: #2a2a9fab;
            overflow: hidden;
            font-weight: 700;
            border: 5px solid #bebe0d;"
     >
Predicting Spam
</div>

<div style="color:white;display:fill;
            background-color:#2a2a9fab;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Load Data</b></p>
</div>

In [5]:
import pandas as pd

yelp = pd.read_csv('/Users/andystorer/Desktop/Grad School-Miami/Classes /Fall 2025/ISA 514/Module 7 - Text Mining/Data/yelp_review.csv')

yelp.head()

,class,review
0,2,"Contrary to other reviews, I have zero complai..."
1,1,Last summer I had an appointment to get new ti...
2,2,"Friendly staff, same starbucks fair you get an..."
3,1,The food is good. Unfortunately the service is...
4,2,Even when we didn't have a car Filene's Baseme...


<div style="color:white;display:fill;
            background-color:#2a2a9fab;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Define Text Corpus</b></p>
</div>

In [6]:
corpus = yelp['review']
corpus.head()

0    Contrary to other reviews, I have zero complai...
1    Last summer I had an appointment to get new ti...
2    Friendly staff, same starbucks fair you get an...
3    The food is good. Unfortunately the service is...
4    Even when we didn't have a car Filene's Baseme...
Name: review, dtype: str

<div style="color:white;display:fill;
            background-color:#2a2a9fab;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Predict All Four Sentiment Scores</b></p>
</div>

In [1]:
# import nltk vader library
from nltk.sentiment.vader import SentimentIntensityAnalyzer

# initiate an analyzer
sia = SentimentIntensityAnalyzer()

senti_pos = []
senti_neg = []
senti_neu = []
senti_comp = []


# iterate through each sentence in corpus
for sentence in corpus:

    #print(sentence)

    # analyze the sentiment. ss is a dictionary
    ss = sia.polarity_scores(sentence)

    # output each sentiment score (neg, neu, pos, compound) in ss
    #print(ss['pos']) # for debugging
    senti_pos.append(ss['pos'])
    senti_neg.append(ss['neg'])
    senti_neu.append(ss['neu'])
    senti_comp.append(ss['compound'])

    # print an empty line as seperator
    #print('\n')

ModuleNotFoundError: No module named 'nltk'

In [ ]:
# adding the list to the dataframe as column using assign(column_name = data)
yelp = yelp.assign(pos = senti_pos, neg = senti_neg, neu = senti_neu, compound = senti_comp)

In [ ]:
X = yelp[['pos', 'neg', 'neu', 'compound']]

NameError: name 'yelp' is not defined

In [ ]:
# select target
y=yelp[['class']]

y.head()

In [ ]:
y = y.values.ravel()

In [ ]:
# load the required library
from sklearn.model_selection import train_test_split

# split data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=200)

In [ ]:
# import the library
from sklearn.ensemble import RandomForestClassifier

# initialize the algorithm
rfc_pos = RandomForestClassifier(random_state=200)
rfc_neg = RandomForestClassifier(random_state=200)
rfc_neu = RandomForestClassifier(random_state=200)
rfc_compound = RandomForestClassifier(random_state=200)

# Generate a new model using training data only
rfc_pos.fit(X_train[['pos']],y_train)
rfc_neg.fit(X_train[['neg']],y_train)
rfc_neu.fit(X_train[['neu']],y_train)
rfc_compound.fit(X_train[['compound']],y_train)

In [ ]:
# load the required libraries
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# make a prediction for the input data
y_pred_pos = rfc_pos.predict(X_test[['pos']])
y_pred_neg = rfc_neg.predict(X_test[['neg']])
y_pred_neu = rfc_neu.predict(X_test[['neu']])
y_pred_compound = rfc_compound.predict(X_test[['compound']])

In [ ]:
print(accuracy_score(y_test, y_pred_pos))
print(classification_report(y_test, y_pred_pos))

In [ ]:
print(accuracy_score(y_test, y_pred_neg))
print(classification_report(y_test, y_pred_neg))

In [ ]:
print(accuracy_score(y_test, y_pred_neu))
print(classification_report(y_test, y_pred_neu))

In [ ]:
print(accuracy_score(y_test, y_pred_compound))
print(classification_report(y_test, y_pred_compound))

<div style="color:white;display:fill;
            background-color:#2a2a9fab;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Generate Normalized TF-IDF for the Whole Corpus with Cleaning</b></p>
</div>

In [ ]:
import nltk
# Install required lexicons for your account
nltk.download('stopwords')
nltk.download('punkt')

In [ ]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

corpus_cleaned = []

for text in corpus:
    # Seperate text into individual words
    tokens = word_tokenize(text)

    # Remove the punctuations and numbers
    tokens = [word for word in tokens if word.isalpha()]

    # Lower the tokens
    tokens = [word.lower() for word in tokens]

    # Remove stopword
    tokens = [word for word in tokens if not word in stopwords.words("english")]

    # Stem the tokens
    ps = PorterStemmer()
    tokens = [ps.stem(w) for w in tokens]

    text_cleaned = " ".join(tokens)

    corpus_cleaned.append(text_cleaned)

In [ ]:
# genereate normalized TF-IDF DTM

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(norm='l2')
X_tfidf = vectorizer.fit_transform(corpus_cleaned)

#print(vectorizer.get_feature_names())
#print(X_tfidf.toarray())

In [ ]:
# covert DTM to a DataFrame
X_tfidf = pd.DataFrame(X_tfidf.toarray())
#X_tfidf.columns=vectorizer.get_feature_names() # sklearn.__version__ <= 0.24.x
X_tfidf.columns=vectorizer.get_feature_names_out() # sklearn.__version__ >= 1.0.x

#X_tfidf.head()

In [ ]:
# load the required library
from sklearn.model_selection import train_test_split

# split data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.20, random_state=200)

In [ ]:
# import the library
from sklearn.ensemble import RandomForestClassifier

# initialize the algorithm
rfc_tfidf=RandomForestClassifier(random_state=200)

# Generate a new model using training data only
rfc_tfidf.fit(X_train,y_train)

In [ ]:
# load the required libraries
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# make a prediction for the input data
y_pred_tfidf = rfc_tfidf.predict(X_test)

In [ ]:
print(accuracy_score(y_test, y_pred_tfidf))
print(classification_report(y_test, y_pred_tfidf))

<div style="color:white;display:fill;
            background-color:#2a2a9fab;font-size:200%;">
    <p style="padding: 4px;color:white;"><b>Use Sentiment & Normalized TF-IDF </b></p>
</div>

In [ ]:
X_full = X_tfidf.assign(pos = senti_pos)

In [ ]:
X_full.head()

In [ ]:
# load the required library
from sklearn.model_selection import train_test_split

# split data into training (70%) and testing (30%)
X_train, X_test, y_train, y_test = train_test_split(X_full, y, test_size=0.20, random_state=200)

In [ ]:
# import the library
from sklearn.ensemble import RandomForestClassifier

# initialize the algorithm
rfc_full = RandomForestClassifier(random_state=200)

# Generate a new model using training data only
rfc_full.fit(X_train,y_train)

In [ ]:
# load the required libraries
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# make a prediction for the input data
y_pred_full = rfc_full.predict(X_test)

In [ ]:
print(accuracy_score(y_test, y_pred_full))
print(classification_report(y_test, y_pred_full))